# Attention Causal Structure

Analyze causal dependencies in attention: causal chains, bottlenecks,
multi-hop paths, influence matrices, and causal structure summary.

## Why This Matters

Causal attention structure applies rigorous causal inference methods to understanding model internals. Causal methods go beyond correlation to establish which components are genuinely responsible for specific behaviors, providing the strongest form of mechanistic evidence.

**Key references:**
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification
- [Conmy et al. (2023) "Towards Automated Circuit Discovery"](https://arxiv.org/abs/2304.14997) — Automated methods for circuit finding via edge pruning

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_causal_structure import (
    causal_attention_chain, attention_information_bottleneck,
    multi_hop_attention_paths, causal_influence_matrix,
    causal_structure_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Causal Attention Chain

Trace the chain of attention back from the last position.

In [ ]:
result = causal_attention_chain(model, tokens, target_position=-1)
print(f"Root positions: {result['root_positions']}")
print(f"N roots: {result['n_root_positions']}\n")
for p in result['per_layer']:
    tracked = ', '.join(f'{k}:{v:.3f}' for k, v in sorted(p['tracked_positions'].items()))
    print(f"  Layer {p['layer']}: {p['n_active_positions']} positions | {tracked}")

## Information Bottleneck

Which positions act as information bottlenecks?

In [ ]:
result = attention_information_bottleneck(model, tokens)
print(f"Top bottleneck: position {result['top_bottleneck_position']}")
print(f"Clear bottleneck: {result['has_clear_bottleneck']}\n")
for p in result['per_position']:
    bar = '█' * int(min(p['bottleneck_score'] * 5, 30))
    print(f"  Pos {p['position']} (tok {p['token_id']:3d}): "
          f"in={p['total_incoming']:.3f}, out={p['total_outgoing']:.3f}, "
          f"score={p['bottleneck_score']:.3f} {bar}")

## Multi-Hop Attention Paths

Find multi-hop paths from source to target through attention layers.

In [ ]:
for src in [0, 1, 2]:
    result = multi_hop_attention_paths(model, tokens, source=src, target=6)
    print(f"  {src} -> 6: max_strength={result['max_path_strength']:.4f}, "
          f"best_depth={result['best_depth']}")
    for d in result['per_depth']:
        bar = '█' * int(d['path_strength'] * 30)
        print(f"    depth {d['depth']}: {d['path_strength']:.4f} {bar}")

## Causal Influence Matrix

Position-to-position causal influence at each layer.

In [ ]:
for layer in range(4):
    result = causal_influence_matrix(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean={result['mean_influence']:.4f}, "
          f"max={result['max_influence']:.4f}")

## Causal Structure Summary

Cross-layer overview of attention causal structure.

In [ ]:
result = causal_structure_summary(model, tokens)
print(f"Overall pattern: {result['overall_pattern']}\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: entropy={p['mean_entropy']:.4f}, "
          f"bos_dom={p['bos_dominant_fraction']:.2%}, "
          f"self={p['self_attention_fraction']:.4f}")